# 11 · Next-step tests: do the eps results survive harder checks?

**What this notebook tests.** Notebooks 01 to 10 left six open questions. Each section below answers one of them.

| # | Question | Test | What would count as a pass |
|---|---|---|---|
| 1 | Is the model reading **behavior** or **location**? (RQ2) | New dataset where every ant in a colony has its **own** eps (10 values mixed in each run), so the colony cannot give eps away. Train A, B and the shuffled control D on it. | Reading with no history is flat (no leak); A still reads eps after a long history; D (no behavior) clearly worse than A. |
| 2 | Do the RQ3 and RQ4 results hold **without the rule as input**? | Probes, eps vectors and the injection knob for model A (shared eps), A_mix (own eps) and B side by side. | A and A_mix show readable eps in mid or late layers and an injection knob clearly better than random vectors. |
| 3 | Is A's reading **pulled to the middle by its rule errors**? | On the own-eps data: a bigger, longer-trained A; a two-step A (learn the rule first, then next moves); and an **ideal observer** that knows the rule and computes the exact posterior over eps from the ant's moves. | Better rule agreement gives a steeper eps reading; the ideal observer shows how much a behavior-only reader can know. |
| 4 | Can a **better readout** recover more than 0.58 of the 0.90 model B knows? | Window-average linear probe (old), all-layer probe, per-step linear probe, per-step MLP probe on hidden + state. | A readout close to the 0.90 ceiling. |
| 5 | Why can't injection reach **eps 0.1 and 1.0**, and must strength be hand-tuned? | Strength up to 8, per-target strengths, and a vector **optimized by gradient** to hit each extreme. | Either some vector reaches the extremes (the mean vector is the limit) or none does (the model itself cannot express them). |
| 6 | Does this work for a trait that changes **which** move is chosen? | New rule: when following smell, an ant trusts the trail with probability **w**, otherwise keeps its previous direction (eps fixed at 0.1). Own w per ant. Read, probe and set w. | A reads w from history with a flat history-0 reading, and an injected w vector sets it. |

**Runtime (GPU, full settings).** About 1 to 1.5 hours: two new datasets (about 10 minutes each), seven transformer trainings (3 to 10 minutes each), the rest a few minutes. Datasets and models are saved and reused on a rerun.

**Needs.** Notebooks 03, 04, 06 and 07 already run (it loads `brain_model.pt`, `seq_A.pt`, `seq_B.pt`, `q2_eps_vectors.npz`) and `ant_common.py` next to this notebook.

**Results summary.** *Fill in after the run: section 7 prints every headline number and a pass/fail line per test.*
- Test 1:
- Test 2:
- Test 3:
- Test 4:
- Test 5:
- Test 6:

**Conclusion.** *Fill in after the run.*

In [1]:
# Shared plumbing (paths, rule, models, plotting) lives in ant_common.py next to this notebook.
from ant_common import *

setup_plots()
seed_everything(0)
print("device:", DEVICE, "| quick mode:", QUICK)
print("data:  ", DATA_DIR)
print("output:", OUT_DIR)

import dataclasses, json, types

RESULTS = {}                                    # every headline number lands here; printed in section 7
TARGETS = [e for e in EPS_GRID if e <= 0.9 + 1e-6 and not np.isclose(e, 0.5)]   # knob targets (source 0.5)
ALPHA_GRID = [1.0, 1.5, 2.0, 2.5, 3.0, 4.0, 5.0]
PROBE_POSITIONS = [0, 7, 31, 63, 127]


def fit_line(x, y):
    """Slope and intercept on finite points only."""
    x, y = np.asarray(x, float), np.asarray(y, float)
    ok = np.isfinite(x) & np.isfinite(y)
    return tuple(np.polyfit(x[ok], y[ok], 1)) if ok.sum() >= 2 else (np.nan, np.nan)


def corr2(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    ok = np.isfinite(a) & np.isfinite(b)
    return np.corrcoef(a[ok], b[ok])[0, 1] ** 2 if ok.sum() > 2 else np.nan

device: cuda | quick mode: False
data:   /home/spatel/Desktop/Projects/collective_intelligence/ant_sim/simple_ant_sim/data/dataset_20261025_104552_105991
output: /home/spatel/Desktop/Projects/collective_intelligence/ant_sim/simple_ant_sim/data/eps_models_v2


## 0. New datasets: every ant has its own trait

The real simulator runs unchanged; we only swap its `decide` for a wrapper that gives each ant its own value.
- **eps dataset**: each ant calls the real rule with its own eps. Each run mixes all 10 eps values (10 ants each with 100 ants), so every colony looks the same on average and cannot give an ant's eps away.
- **w dataset (test 6)**: a new trail-trust rule. With probability 0.1 a random move; if the target is visible, go to it; if a smell points somewhere, follow it with probability **w**, otherwise keep the previous direction; with no signal, a random move. Each ant has its own w.

Same file format as the original dataset, plus `ant_trait` (one value per ant). Seeds do not overlap the original dataset.

In [ ]:
NEW_DATA = OUT_DIR / "datasets"
N_ANTS = 30 if QUICK else 100                   # must be a multiple of 10
T_RUN = 1500
SPLIT_SEEDS = {"train": range(4 if QUICK else 60), "val": range(2 if QUICK else 20), "test": range(2 if QUICK else 20)}
SEED_BASE = {"eps": 10_000, "w": 20_000}
SPLIT_OFFSET = {"train": 0, "val": 1000, "test": 2000}
EPS_W = 0.1                                     # fixed random-move rate in the w rule

sim_mod, Simulation, Config = import_sim()
original_decide = sim_mod.decide


def trait_decider(trait, values, cfg):
    """A decide() in which ant i uses values[i] as its eps (trait 'eps') or its trail trust w (trait 'w')."""
    if trait == "eps":
        cfgs = [dataclasses.replace(cfg, epsilon=float(v)) for v in values]
        return lambda ant, local, rng, _cfg: original_decide(ant, local, rng, cfgs[ant.id])

    last = {}                                                   # previous direction per ant

    def decide_w(ant, local, rng, _cfg):
        carrying = ant.has_food()
        target = np.asarray(local.see_home if carrying else local.see_food)
        smell = np.asarray(local.smell_home if carrying else local.smell_food)
        d = None
        if rng.random() < EPS_W:
            d = int(rng.integers(N_ACTIONS))                    # random move
        elif target.max() > 0:
            d = int(rng.choice(np.flatnonzero(target == target.max())))     # go to the target
        elif smell.max() > smell.min():                         # a smell points somewhere
            if rng.random() < values[ant.id]:
                d = int(rng.choice(np.flatnonzero(smell == smell.max())))   # trust the trail
            elif ant.id in last:
                d = last[ant.id]                                # keep going the same way
        if d is None:
            d = int(rng.integers(N_ACTIONS))
        last[ant.id] = d
        action = np.zeros(N_ACTIONS, dtype=np.float32)
        action[d] = 1.0
        return types.SimpleNamespace(action=action, pick=bool(not carrying and local.on_food),
                                     drop=bool(carrying and local.at_home), mode=0)
    return decide_w


def generate_run(trait, values, seed, path):
    cfg = Config(seed=seed, n_ants=N_ANTS)
    sim = Simulation(cfg)
    for x, y in FOOD_PATCHES:
        sim.world.add_food(x, y, radius=4, amount_per_cell=5)
    inner, rec = trait_decider(trait, values, cfg), {}

    def logging_decide(ant, local, rng, c):
        dec = inner(ant, local, rng, c)
        rec[ant.id] = (np.asarray(ant.obs, dtype=np.float32).copy(), int(np.argmax(dec.action)),
                       bool(dec.pick), bool(dec.drop))
        return dec

    S = np.zeros((T_RUN, N_ANTS, 19), np.float32)
    A = np.zeros((T_RUN, N_ANTS), np.int64)
    P = np.zeros((T_RUN, N_ANTS), bool)
    D = np.zeros((T_RUN, N_ANTS), bool)
    delivered = [0]
    sim_mod.decide = logging_decide
    try:
        for t in range(T_RUN):
            sim.step()
            for i in range(N_ANTS):
                S[t, i], A[t, i], P[t, i], D[t, i] = rec[i]
            delivered.append(sim.world.food_delivered)
    finally:
        sim_mod.decide = original_decide                        # always restore the real brain
    prev = np.zeros((T_RUN, N_ANTS, 4), np.float32)
    prev[1:] = np.eye(4, dtype=np.float32)[A[:-1]]
    path.mkdir(parents=True, exist_ok=True)
    np.savez(path / "traj.npz", state=S, action=A, prev_action=prev, dec_pick=P, dec_drop=D,
             delivered=np.array(delivered), epsilon=float(np.mean(values)), seed=seed,
             ant_trait=np.asarray(values, np.float32))


def make_dataset(trait):
    """Generates (or reuses) a dataset where every run mixes all 10 trait values."""
    root = NEW_DATA / f"mixed_{trait}_a{N_ANTS}"
    t0, made = time.time(), 0
    for split, seeds in SPLIT_SEEDS.items():
        for k in seeds:
            seed = SEED_BASE[trait] + SPLIT_OFFSET[split] + k
            path = root / split / f"seed{seed:05d}"
            if (path / "traj.npz").exists():
                continue
            values = np.random.default_rng(seed).permutation(np.repeat(EPS_GRID, N_ANTS // len(EPS_GRID)))
            generate_run(trait, values, seed, path)
            made += 1
    print(f"{trait}: {root}  ({made} new runs, {time.time() - t0:.0f} s)")
    return root


def load_trait_split(root, split):
    runs = []
    for p in sorted((root / split).glob("*/traj.npz")):
        r = load_run(p.parent)
        with np.load(p) as d:
            r["ant_trait"] = d["ant_trait"].astype(np.float32)
        runs.append(r)
    return runs


def trait_sequences(runs, **opts):
    """build_sequences, but the label per ant is that ant's own trait value."""
    X, Y, _ = build_sequences(runs, **opts)
    return X, Y, np.concatenate([r["ant_trait"] for r in runs])


eps_root = make_dataset("eps")
mix = {split: load_trait_split(eps_root, split) for split in ["train", "val", "test"]}
print({split: len(r) for split, r in mix.items()}, "runs")

In [ ]:
# Check: within each run, every ant follows the rule at its OWN rate (1 - eps) + eps/4
rows = []
for r in mix["train"]:
    s, a = r["state"], r["action"]                              # (T, N, 19), (T, N)
    for i, e in enumerate(r["ant_trait"]):
        si, ai = s[:, i], a[:, i]
        clear = n_ties(si) == 1
        rows.append(dict(eps=round(float(e), 1), follows=safe_mean(ai[clear] == best_dir(si)[clear]),
                         share_clear=clear.mean()))
own = pd.DataFrame(rows).groupby("eps").mean()
own["expected"] = 1 - own.index + own.index / 4
delivered = [r["delivered"][-1] for r in mix["train"]]
print(own.round(3))
print(f"\nfood delivered per run: mean {np.mean(delivered):.0f}, std {np.std(delivered):.0f} (every colony is the same mix)")

fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
axes[0].plot(own.index, own.expected, ls="--", color=C["gray"], lw=1.2, label="expected (1 - eps) + eps/4")
axes[0].plot(own.index, own.follows, "o", color=C["blue"], label="measured, per ant")
axes[0].set(xlabel="the ant's own eps", ylabel="follows rule (clear steps)", title="Each ant keeps its own eps")
axes[0].legend()
axes[1].plot(own.index, own.share_clear, "-o", color=C["blue"])
axes[1].set(xlabel="the ant's own eps", ylabel="share of clear-choice steps", ylim=(0, 1),
            title="Does an ant's own eps change what it sees?")
plt.tight_layout()
plt.show()

**How to read this.** Left: every ant should sit on the dashed line, which confirms the wrapper gives each ant its own eps. Right: in the shared-eps data the share of clear steps fell from 0.64 to 0.20 because the *whole colony* changed. Here colonies are identical, so any slope left on the right is an *individual* location effect (a random ant wanders off trails), which is legitimately about that ant.

*Analysis: fill in after the run.*

## 1. Behavior or location? Models on the own-eps data

Three models, same architecture and training as notebooks 04 to 06: **A_mix** (states + previous moves), **D_mix** (states, time shuffled: no behavior), **B_mix** (A_mix + the brain's odds). Reference: model **A** on the original shared-eps data.

In [ ]:
brain = load_brain()
VARIANTS = {
    "A_mix": dict(with_actions=True),
    "D_mix": dict(with_actions=False, shuffle=True),
    "B_mix": dict(with_actions=True, brain=brain),
}
models, preds = {}, {}
for name, opts in VARIANTS.items():
    path = OUT_DIR / f"seq_{name}.pt"
    if path.exists():
        models[name] = load_seq_model(name)
        print(f"== {name}: loaded {path.name}")
    else:
        print(f"== {name}")
        seed_everything(0)
        train = trait_sequences(mix["train"], **opts)
        model = AntGPT(d_in=train[0].shape[-1]).to(DEVICE)
        train_seq_model(model, train)
        save_seq_model(model, name, d_in=train[0].shape[-1])
        del train
        models[name] = model
    preds[name] = collect_seq(models[name], trait_sequences(mix["val"], **opts))

model_A = load_seq_model("A")                                   # notebook 04, shared-eps data
orig_val_A = build_sequences(load_split("val"), with_actions=True)
preds["A (shared eps)"] = collect_seq(model_A, orig_val_A)

results1 = {name: summarize_seq_model(*p) for name, p in preds.items()}
headline = pd.DataFrame({name: r[2] for name, r in results1.items()})
for name, (_, by_eps, _) in results1.items():
    headline.loc["history-0 slope (leak, want 0)", name] = fit_line(by_eps.index, by_eps["0"])[0]
print(headline.round(4))
print("\nimplied eps with NO history:")
print(pd.DataFrame({n: r[1]["0"] for n, r in results1.items()}).round(3))
print("\nimplied eps after 64-127 steps:")
print(pd.DataFrame({n: r[1]["64-127"] for n, r in results1.items()}).round(3))

for name in results1:
    RESULTS[f"T1 {name}: eps error (<=0.7)"] = results1[name][2]["eps error, eps<=0.7"]
    RESULTS[f"T1 {name}: slope late / history 0"] = (results1[name][2]["eps slope (want 1)"],
                                                      headline.loc["history-0 slope (leak, want 0)", name])

In [ ]:
colors = {"A (shared eps)": C["gray"], "A_mix": C["blue"], "D_mix": C["aqua"], "B_mix": C["orange"]}
styles = {"A (shared eps)": "--"}
fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
for ax, col, title in [(axes[0], "0", "No history: is there still a leak?"),
                       (axes[1], "64-127", "After a long history")]:
    diagonal(ax)
    for name, (_, by_eps, _) in results1.items():
        ax.plot(by_eps.index, by_eps[col], marker="o", ls=styles.get(name, "-"), color=colors[name], label=name)
    ax.set(xlabel="true eps (the ant's own)", ylabel=f"implied eps (history {col})", title=title)
    ax.legend(fontsize=8)
ax = axes[2]
names = list(results1)
ax.bar(np.arange(len(names)), [results1[n][2]["eps error, eps<=0.7"] for n in names],
       color=[colors[n] for n in names], width=0.6)
ax.set_xticks(np.arange(len(names)), names, fontsize=9)
ax.set(ylabel="mean |implied - true| (eps <= 0.7)", title="Reading error after a long history")
plt.tight_layout()
plt.show()

**How to read this.**
- **Left (leak).** With shared eps, model A's history-0 line rose with eps. If the A_mix, D_mix and B_mix lines are flat here, the colony-level leak is gone by design.
- **Middle and right (behavior).** If A_mix still climbs toward the diagonal while D_mix stays much flatter, the reading comes from how the ant moves: a real RQ1 and RQ2 result. If D_mix reads eps almost as well as A_mix, an ant's own location still gives its eps away (random ants end up in different places), which is individual but not behavioral.
- **A_mix vs A (shared eps).** A drop in accuracy measures how much the old result relied on the colony leak.

*Analysis: fill in after the run.*

## 2. Repeat RQ3 and RQ4 on models that are not given the rule

For each model: linear probes per stage (val → test), the eps vectors (average hidden state per eps), the injection layer and strength chosen **on validation**, and the knob from eps 0.5 windows on **test**, against a random vector of the same length. Model B is rerun for a side-by-side comparison. Knob targets are 0.1 to 0.9 without 0.5 (the source), so B's number differs slightly from notebook 08.

In [ ]:
def analyse(model, data_val, data_test):
    """Probes, eps vectors and the injection knob for one model. Returns a dict of results."""
    val_h = collect_hidden(model, data_val, seed=0)
    test_h = collect_hidden(model, data_test, seed=1)
    probe = pd.DataFrame({stage: [r2(test_h["eps"], ridge_fit(val_h["hidden"][k][:, t].astype(np.float64),
                                                               val_h["eps"].astype(np.float64))(
                                          test_h["hidden"][k][:, t].astype(np.float64)))
                                  for t in PROBE_POSITIONS]
                          for k, stage in enumerate(STAGES)}, index=PROBE_POSITIONS).T
    ceiling = corr2(test_h["eps"], implied_eps_windows(test_h["probs"], test_h["states"]))
    means = {k: eps_means(val_h["hidden"][k], val_h["eps"]) for k in range(len(STAGES))}

    goals_val = natural_reading(model, data_val, seed=10)
    stage_err = {k: knob_error(model, means[k], data_val, goals_val, 2.0, 0.5, TARGETS, stage=k, seed=10)
                 for k in (1, 2, 3)}
    stage = min(stage_err, key=stage_err.get)
    alpha_err = {a: knob_error(model, means[stage], data_val, goals_val, a, 0.5, TARGETS, stage=stage, seed=10)
                 for a in ALPHA_GRID}
    alpha = min(alpha_err, key=alpha_err.get)

    goals = natural_reading(model, data_test, seed=0)
    x_s, s_s = windows_for_eps(data_test, 0.5, seed=0)
    rng = np.random.default_rng(0)
    rows = []
    for t in EPS_GRID:
        v = alpha * (means[stage][t] - means[stage][0.5])
        p = run_with_injection(model, x_s, stage, v)
        rows.append(dict(target=t, goal=goals[t], injected=safe_nanmean(implied_eps_windows(p, s_s)),
                         random=safe_nanmean(implied_eps_windows(run_with_injection(model, x_s, stage, random_like(v, rng)), s_s)),
                         rule=rule_agreement(p, s_s)))
    knob = pd.DataFrame(rows).set_index("target")
    on = knob.loc[TARGETS]
    return dict(val_h=val_h, test_h=test_h, probe=probe, ceiling=ceiling, means=means, stage=stage,
                stage_err=stage_err, alpha=alpha, alpha_err=alpha_err, goals=goals, knob=knob,
                test_err=np.abs(on.injected - on.goal).mean(), random_err=np.abs(on.random - on.goal).mean(),
                min_rule=on.rule.min())


orig = {split: load_split(split) for split in ["val", "test"]}
SETS = {
    "B":     (load_seq_model("B"), {s: build_sequences(orig[s], with_actions=True, brain=brain) for s in orig}),
    "A":     (model_A,             {s: build_sequences(orig[s], with_actions=True) for s in orig}),
    "A_mix": (models["A_mix"],     {s: trait_sequences(mix[s], with_actions=True) for s in ["val", "test"]}),
}
analyses = {}
for name, (model, data) in SETS.items():
    t0 = time.time()
    analyses[name] = analyse(model, data["val"], data["test"])
    a = analyses[name]
    print(f"{name:6s} probe R² (layer 3, t=127) {a['probe'].iloc[-1, -1]:.3f} | ceiling {a['ceiling']:.3f} | "
          f"stage {a['stage']} | strength {a['alpha']} | knob test error {a['test_err']:.3f} "
          f"(random {a['random_err']:.3f}) | min rule agreement {a['min_rule']:.3f}  ({time.time() - t0:.0f} s)")
    RESULTS[f"T2 {name}: probe R2 layer3 t127 / ceiling"] = (a["probe"].iloc[-1, -1], a["ceiling"])
    RESULTS[f"T2 {name}: knob error / random (stage {a['stage']}, strength {a['alpha']})"] = (a["test_err"], a["random_err"])

print()
for name, a in analyses.items():
    print(f"== {name}: probe R² (test)")
    print(a["probe"].round(3))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 9))
for j, (name, a) in enumerate(analyses.items()):
    ax = axes[0, j]
    for stage, color in zip(STAGES, RAMP[1:]):
        ax.plot(PROBE_POSITIONS, a["probe"].loc[stage].values, "-o", color=color, label=stage)
    ax.axhline(a["ceiling"], ls="--", color=C["gray"], lw=1.2, label=f"model output corr² = {a['ceiling']:.2f}")
    ax.set(xlabel="steps of history seen", ylabel="probe R² (test)", ylim=(-0.1, 1), title=f"Model {name}: where eps is readable")
    ax.legend(fontsize=8)

    ax = axes[1, j]
    k = a["knob"]
    ax.plot(k.index, k.goal, "-o", color=C["aqua"], label="goal (natural reading)")
    ax.plot(k.index, k.injected, "-o", color=C["blue"], label=f"eps 0.5 + eps vector (stage {a['stage']}, x{a['alpha']})")
    ax.plot(k.index, k.random, "--s", color=C["orange"], label="eps 0.5 + random vector")
    ax.set(xlabel="target eps", ylabel="implied eps (test)", ylim=(0, 1.05),
           title=f"Model {name}: knob error {a['test_err']:.3f} (random {a['random_err']:.3f})")
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

**How to read this.**
- **Top row.** If A and A_mix show probe R² rising in layers 2 or 3 like B does, the internal eps estimate is not an artifact of being given the rule.
- **Bottom row.** If the blue line follows the green goal and beats the orange random control for A_mix, the knob works on a model that learned everything from behavior alone, with no leak. That is the strongest version of the RQ4 claim.
- **Compare A with A_mix.** If A's probe or knob is much better than A_mix's, part of A's eps signal was the colony leak.

*Analysis: fill in after the run.*

## 3. Does A's rule gap pull its reading to the middle? Bigger A, two-step A, and an ideal observer

All on the own-eps data, scored on the same validation windows as A_mix and B_mix.
- **A_big**: width 128, 4 layers, 30 epochs.
- **A_2step**: same size as A. First trained to predict the **rule's** preferred move on clear steps (taken from the state, no eps), then on next moves as usual. Never sees eps.
- **Ideal observer**: knows the rule exactly and the 10 possible eps values, and computes the exact posterior over this ant's eps from its moves so far. It uses only *how the ant moved relative to the rule*, so it is the best possible behavior-only reader for these windows.

In [ ]:
def train_on_targets(model, X, TGT, epochs, batch=SEQ_BATCH, lr=SEQ_LR):
    """Like train_seq_model, but with any per-step targets (-100 = ignore)."""
    ants, t0 = all_windows(len(X), X.shape[1])
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    for epoch in range(1, epochs + 1):
        order, total, count, start_time = np.random.permutation(len(ants)), 0.0, 0, time.time()
        for start in range(0, len(order), batch):
            sel = order[start:start + batch]
            x, y, _ = get_batch(X, TGT, np.zeros(len(X)), ants[sel], t0[sel])
            loss = F.cross_entropy(model(x.to(DEVICE)).reshape(-1, N_ACTIONS), y.reshape(-1).to(DEVICE), ignore_index=-100)
            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            total, count = total + loss.item(), count + 1
        print(f"   rule epoch {epoch:>2}: loss {total / count:.4f}   ({time.time() - start_time:.0f} s)")
    model.eval()


def load_or_train(name, build, train_fn):
    path = OUT_DIR / f"seq_{name}_full.pt"
    model = build()
    if path.exists():
        model.load_state_dict(torch.load(path, map_location=DEVICE))
        print(f"== {name}: loaded {path.name}")
    else:
        print(f"== {name}")
        seed_everything(0)
        train_fn(model)
        torch.save(model.state_dict(), path)
    return model.eval()


train_A = trait_sequences(mix["train"], with_actions=True)
val_A = trait_sequences(mix["val"], with_actions=True)


def train_big(model):
    train_seq_model(model, train_A, epochs=1 if QUICK else 30)


def train_two_step(model):
    S = train_A[0][..., :19].reshape(-1, 19)
    rule_tgt = np.where(n_ties(S) == 1, best_dir(S), -100).reshape(train_A[1].shape)   # the rule's move, clear steps only
    train_on_targets(model, train_A[0], rule_tgt, epochs=1 if QUICK else 5)
    train_seq_model(model, train_A)


models["A_big"] = load_or_train("A_big", lambda: AntGPT(d_in=23, d_model=128, n_layers=4).to(DEVICE), train_big)
models["A_2step"] = load_or_train("A_2step", lambda: AntGPT(d_in=23).to(DEVICE), train_two_step)
del train_A
preds["A_big"] = collect_seq(models["A_big"], val_A)
preds["A_2step"] = collect_seq(models["A_2step"], val_A)
mix.pop("train")                                                # free memory

In [ ]:
def ideal_observer(states, acts, grid=EPS_GRID):
    """Exact Bayesian reader: knows the rule and the possible eps values, uniform prior.
    Returns its next-move probabilities (W, L, 4) and its posterior-mean eps (W, L)."""
    W, L, _ = states.shape
    s = states.reshape(-1, 19)
    P = np.stack([true_probs(s, e).reshape(W, L, 4) for e in grid])                 # (E, W, L, 4)
    lik = np.take_along_axis(P, np.broadcast_to(acts[None, ..., None], (len(grid), W, L, 1)), axis=-1)[..., 0]
    cum = np.cumsum(np.log(lik + 1e-12), axis=2)
    before = np.concatenate([np.zeros((len(grid), W, 1)), cum[:, :, :-1]], axis=2)  # evidence from steps < t
    post = np.exp(before - before.max(0, keepdims=True))
    post /= post.sum(0, keepdims=True)
    return (post[..., None] * P).sum(0).astype(np.float32), (post * np.asarray(grid)[:, None, None]).sum(0)


_, acts_v, states_v, eps_v = preds["A_mix"]
io_probs, io_mean = ideal_observer(states_v, acts_v)
preds["ideal observer"] = (io_probs, acts_v, states_v, eps_v)

compare = ["A_mix", "A_big", "A_2step", "B_mix", "ideal observer"]
results3 = {name: summarize_seq_model(*preds[name]) for name in compare}
print(pd.DataFrame({n: r[2] for n, r in results3.items()}).round(4))
print("\nrule agreement by history:")
print(pd.DataFrame({n: r[0]["rule_agreement"] for n, r in results3.items()}).round(4))
io_err = np.abs(io_mean[:, LATE].mean(1) - eps_v)
print(f"\nideal observer posterior mean, steps 64-127: mean |error| {io_err.mean():.3f}")
for name in compare:
    sc = results3[name][2]
    RESULTS[f"T3 {name}: rule agreement / eps error / slope"] = (sc["rule agreement, 64-127 (eps<=0.7)"],
                                                                 sc["eps error, eps<=0.7"], sc["eps slope (want 1)"])

In [ ]:
colors3 = {"A_mix": C["blue"], "A_big": C["magenta"], "A_2step": C["green"], "B_mix": C["orange"], "ideal observer": C["ink"]}
fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
for name, (by_hist, by_eps, sc) in results3.items():
    ls = "--" if name == "ideal observer" else "-"
    axes[0].plot(by_hist.index, by_hist.gap, marker="o", ls=ls, color=colors3[name], label=name)
    axes[1].plot(by_eps.index, by_eps["64-127"], marker="o", ls=ls, color=colors3[name], label=name)
    axes[2].plot(by_hist.index, by_hist.rule_agreement, marker="o", ls=ls, color=colors3[name], label=name)
axes[0].axhline(0, color=C["ink"], lw=0.6)
axes[0].set(xlabel="history range (steps)", ylabel="loss above floor", title="How close to the best possible")
diagonal(axes[1])
axes[1].set(xlabel="true eps", ylabel="implied eps (history 64-127)", title="Reading after a long history")
axes[2].set(xlabel="history range (steps)", ylabel="agrees with rule (clear steps, eps <= 0.7)", title="How well the rule is known")
for ax in axes:
    ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

**How to read this.**
- **Ideal observer vs B_mix.** The ideal observer is the ceiling for reading eps from behavior. Its own reading is also squeezed toward the middle with only 64 to 127 steps of evidence; that part of the compression is unavoidable, not a model flaw.
- **A_big and A_2step vs A_mix.** If higher rule agreement comes with a steeper, more accurate eps reading, A's rule errors were pulling it to the middle. If the readings barely move, the limit is evidence, not the rule.
- **Gap to the ideal observer** is how much any model still leaves on the table.

*Analysis: fill in after the run.*

## 4. A better readout: can we recover what model B knows?

Notebook 07 found a window-average linear probe reaches corr² 0.58 while B's own output reaches 0.90. Four readouts, all fit on val windows and scored on test windows, for B and A_mix:
1. **Window mean, linear** (the old one): average layer-3 state over steps 64 to 127.
2. **Window mean, all layers**: layers 1 to 3 concatenated.
3. **Per step, linear**: a linear readout at every clear-choice step, then averaged over the window (the way the model's own output is read).
4. **Per step, MLP**: a small network on the layer-3 state *plus the 19-number state* at every clear step, then averaged.

In [ ]:
def per_step(h, k=3):
    """Layer-k state, raw state and clear-step mask at long-history positions."""
    H = h["hidden"][k][:, LATE].astype(np.float32)                                  # (W, 64, 64)
    S = h["states"][:, LATE]                                                        # (W, 64, 19)
    clear = (n_ties(S.reshape(-1, 19)) == 1).reshape(S.shape[:2])
    return H, S, clear


def window_average(pred_steps, clear):
    n = clear.sum(1)
    return np.where(n > 0, (pred_steps * clear).sum(1) / np.maximum(n, 1), np.nan)


def mlp_probe(X, y, epochs=3 if QUICK else 15, hidden=128):
    net = nn.Sequential(nn.Linear(X.shape[1], hidden), nn.ReLU(), nn.Linear(hidden, hidden), nn.ReLU(),
                        nn.Linear(hidden, 1)).to(DEVICE)
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    Xt, yt = torch.from_numpy(X).float(), torch.from_numpy(y).float()
    for _ in range(epochs):
        perm = torch.randperm(len(Xt))
        for start in range(0, len(Xt), 1024):
            idx = perm[start:start + 1024]
            loss = F.mse_loss(net(Xt[idx].to(DEVICE)).squeeze(-1), yt[idx].to(DEVICE))
            opt.zero_grad()
            loss.backward()
            opt.step()
    net.eval()
    return lambda Xn: net(torch.from_numpy(Xn).float().to(DEVICE)).squeeze(-1).detach().cpu().numpy()


readout_rows = []
for name in ["B", "A_mix"]:
    vh, th = analyses[name]["val_h"], analyses[name]["test_h"]
    yv, yt = vh["eps"].astype(np.float64), th["eps"]
    row = {"model": name, "ceiling (model output)": analyses[name]["ceiling"]}

    mean3 = lambda h: h["hidden"][3][:, LATE].astype(np.float64).mean(1)
    row["window mean, linear"] = corr2(yt, ridge_fit(mean3(vh), yv)(mean3(th)))
    mean_all = lambda h: np.concatenate([h["hidden"][k][:, LATE].astype(np.float64).mean(1) for k in (1, 2, 3)], axis=1)
    row["window mean, all layers"] = corr2(yt, ridge_fit(mean_all(vh), yv)(mean_all(th)))

    Hv, Sv, cv = per_step(vh)
    Ht, St, ct = per_step(th)
    yv_steps = np.broadcast_to(yv[:, None], cv.shape)[cv]
    lin = ridge_fit(Hv[cv].astype(np.float64), yv_steps)
    row["per step, linear"] = corr2(yt, window_average(lin(Ht.reshape(-1, 64)).reshape(ct.shape), ct))
    feats = lambda H, S: np.concatenate([H, S], axis=-1).reshape(-1, 64 + 19)
    mlp = mlp_probe(feats(Hv, Sv)[cv.reshape(-1)], yv_steps.astype(np.float32))
    row["per step, MLP + state"] = corr2(yt, window_average(mlp(feats(Ht, St).astype(np.float32)).reshape(ct.shape), ct))
    readout_rows.append(row)
    RESULTS[f"T4 {name}: best readout / ceiling"] = (max(v for k, v in row.items() if k not in ("model", "ceiling (model output)")),
                                                      row["ceiling (model output)"])

readouts = pd.DataFrame(readout_rows).set_index("model")
print("corr² with true eps on test windows:")
print(readouts.round(3))

fig, ax = plt.subplots(figsize=(11, 4.5))
cols = [c for c in readouts.columns if c != "ceiling (model output)"]
x = np.arange(len(cols))
for i, (name, color) in enumerate(zip(readouts.index, [C["orange"], C["blue"]])):
    ax.bar(x + (i - 0.5) * 0.36, readouts.loc[name, cols], width=0.34, color=color, label=name)
    ax.axhline(readouts.loc[name, "ceiling (model output)"], color=color, ls="--", lw=1.2,
               label=f"{name}: model's own output")
ax.set_xticks(x, cols)
ax.set(ylabel="corr² with true eps (test)", ylim=(0, 1), title="Which readout recovers what the model knows?")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

**How to read this.** If the per-step readouts close most of the gap to the dashed line while the window-mean ones do not, the missing 0.32 was not hidden: averaging hidden states over steps that mean different things (clear vs no-signal steps) threw it away. If only the MLP with the state closes it, eps is stored *in combination with* what the ant currently sees, as a confidence that depends on the situation, not as a free-standing number.

*Analysis: fill in after the run.*

## 5. Extremes and strength (model B, injection after layer 2)

1. **Push strength to 8** toward eps 0.1, 0.2, 0.9 and 1.0 from eps 0.5 windows: is there a ceiling?
2. **Per-target strength** chosen on val vs one strength for all targets.
3. **An optimized vector**: start from the average-difference vector and adjust its 64 numbers by gradient descent on val windows so the implied eps hits the goal. If even this cannot reach eps 1.0 or 0.1, the limit is in what layer 2 can express, not in how we built the vector.

In [ ]:
model_B = SETS["B"][0]
data_B = SETS["B"][1]
MEANS_B = analyses["B"]["means"][2]
goals_test_B = analyses["B"]["goals"]
goals_val_B = natural_reading(model_B, data_B["val"], seed=10)
STRENGTHS = [1, 2, 3, 4, 5, 6, 8]
EXTREMES = [0.1, 0.2, 0.9, 1.0]

x_s, s_s = windows_for_eps(data_B["test"], 0.5, seed=0)
rows = []
for t in EXTREMES:
    for a in STRENGTHS:
        p = run_with_injection(model_B, x_s, 2, a * (MEANS_B[t] - MEANS_B[0.5]))
        rows.append(dict(target=t, strength=a, implied=safe_nanmean(implied_eps_windows(p, s_s)), rule=rule_agreement(p, s_s)))
push = pd.DataFrame(rows)
print(push.pivot(index="strength", columns="target", values="implied").round(3))
print("\nrule agreement:")
print(push.pivot(index="strength", columns="target", values="rule").round(3))

# per-target strength (val) vs one strength (val), scored on test
xv, sv = windows_for_eps(data_B["val"], 0.5, seed=10)
per_target = {}
for t in TARGETS:
    errs = {a: abs(safe_nanmean(implied_eps_windows(run_with_injection(model_B, xv, 2, a * (MEANS_B[t] - MEANS_B[0.5])), sv)) - goals_val_B[t])
            for a in STRENGTHS}
    per_target[t] = min(errs, key=errs.get)
one = {a: knob_error(model_B, MEANS_B, data_B["val"], goals_val_B, a, 0.5, TARGETS, stage=2, seed=10) for a in STRENGTHS}
one_alpha = min(one, key=one.get)
err_one = knob_error(model_B, MEANS_B, data_B["test"], goals_test_B, one_alpha, 0.5, TARGETS, stage=2, seed=0)
err_per = np.mean([abs(safe_nanmean(implied_eps_windows(run_with_injection(model_B, x_s, 2, per_target[t] * (MEANS_B[t] - MEANS_B[0.5])), s_s)) - goals_test_B[t])
                   for t in TARGETS])
print(f"\nbest strength per target (val): {per_target}")
print(f"test error: one strength ({one_alpha}) {err_one:.3f} | per-target strengths {err_per:.3f}")
RESULTS["T5 knob error: one strength / per-target"] = (err_one, err_per)

In [ ]:
def forward_with_vec(model, x, stage, v):
    """Differentiable version of run_with_injection (gradient flows into v only)."""
    T = x.shape[1]
    h = model.inp(x) + model.pos(torch.arange(T, device=x.device))
    mask = nn.Transformer.generate_square_subsequent_mask(T, device=x.device)
    for i, layer in enumerate(model.blocks.layers, start=1):
        h = layer(h, src_mask=mask, is_causal=True)
        if i == stage:
            h = h + v
    return model.out(model.norm(h)).softmax(-1)


def optimize_vector(model, x, states, init, goal, stage=2, steps=40 if QUICK else 300, lr=0.05):
    """Adjust a 64-number vector so the implied eps on these windows hits the goal."""
    for p in model.parameters():
        p.requires_grad_(False)
    W, L, _ = states.shape
    s = states.reshape(-1, 19)
    clear = torch.from_numpy(((n_ties(s) == 1).reshape(W, L)[:, LATE]).astype(np.float32)).to(DEVICE)
    best = torch.from_numpy(best_dir(s).reshape(W, L)[:, LATE]).to(DEVICE)
    v = torch.tensor(init, dtype=torch.float32, device=DEVICE, requires_grad=True)
    opt = torch.optim.Adam([v], lr=lr)
    for _ in range(steps):
        probs = forward_with_vec(model, x, stage, v)[:, LATE]
        p_best = probs.gather(-1, best[..., None])[..., 0]
        implied = ((1 - p_best) / 0.75 * clear).sum() / clear.sum().clamp(min=1)
        loss = (implied - goal) ** 2
        opt.zero_grad()
        loss.backward()
        opt.step()
    for p in model.parameters():
        p.requires_grad_(True)
    return v.detach().cpu().numpy()


model_B.eval()
rows = []
for t in [0.1, 1.0, 0.8]:                                        # 0.8 = a target the mean vector already reaches
    base = one_alpha * (MEANS_B[t] - MEANS_B[0.5])
    v_opt = optimize_vector(model_B, xv, sv, base, goals_val_B[t])
    for kind, v in [("mean vector", base), ("optimized", v_opt)]:
        p = run_with_injection(model_B, x_s, 2, v)
        rows.append(dict(target=t, kind=kind, goal=goals_test_B[t], implied=safe_nanmean(implied_eps_windows(p, s_s)),
                         rule=rule_agreement(p, s_s), length=np.linalg.norm(v),
                         cos_to_mean=float(v @ base / (np.linalg.norm(v) * np.linalg.norm(base) + 1e-12))))
opt_table = pd.DataFrame(rows).set_index(["target", "kind"])
print(opt_table.round(3))
for t in [0.1, 1.0]:
    RESULTS[f"T5 eps {t}: goal / mean vector / optimized"] = (goals_test_B[t], opt_table.loc[(t, "mean vector"), "implied"],
                                                             opt_table.loc[(t, "optimized"), "implied"])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
for t, color in zip(EXTREMES, [C["blue"], C["aqua"], C["orange"], C["magenta"]]):
    part = push[push.target == t]
    axes[0].plot(part.strength, part.implied, "-o", color=color, label=f"target {t}")
    axes[0].axhline(goals_test_B[t], color=color, ls=":", lw=1.2)
    axes[1].plot(part.strength, part.rule, "-o", color=color, label=f"target {t}")
axes[0].set(xlabel="vector strength", ylabel="implied eps (test)", ylim=(0, 1.05),
            title="Pushing harder (dotted = goal)")
axes[1].set(xlabel="vector strength", ylabel="agrees with rule (clear steps)", title="Does the model break?")
for ax in axes[:2]:
    ax.legend(fontsize=8)

ax = axes[2]
labels = [f"{t}" for t in [0.1, 1.0, 0.8]]
xs = np.arange(3)
for i, (kind, color) in enumerate([("mean vector", C["gray"]), ("optimized", C["blue"])]):
    ax.bar(xs + (i - 0.5) * 0.36, [opt_table.loc[(t, kind), "implied"] for t in [0.1, 1.0, 0.8]],
           width=0.34, color=color, label=kind)
ax.scatter(xs, [goals_test_B[t] for t in [0.1, 1.0, 0.8]], marker="_", s=700, lw=2.5, color=C["ink"], zorder=3, label="goal")
ax.set_xticks(xs, labels)
ax.set(xlabel="target eps", ylabel="implied eps (test)", ylim=(0, 1.25), title="Can any layer-2 vector reach the extremes?")
ax.legend(fontsize=8, loc="upper center", ncol=3)
plt.tight_layout()
plt.show()

**How to read this.**
- **Left.** If the line toward 1.0 flattens below its goal however hard we push, the average-difference direction runs out. If it overshoots past some strength, the hand-tuned strength was the only problem.
- **Middle.** Strength that drops rule agreement is breaking the model, not setting eps.
- **Right.** If the optimized vector reaches the extremes (on test windows it never trained on) while keeping rule agreement, layer 2 *can* express them and the mean vector is just a poor direction there. If it cannot, the extremes are not reachable through layer 2 at all.
- **Per-target vs one strength.** A large gain from per-target strengths means the strength depends on distance to the target and should be learned or scheduled, not fixed.

*Analysis: fill in after the run.*

## 6. A trait that changes *which* move is chosen: trail trust w

New rule (eps fixed at 0.1): when a smell points somewhere, the ant follows it with probability **w**, otherwise keeps its previous direction. Each ant has its own w. Unlike eps, w is not "how often the ant breaks the rule": it decides **which** of two specific moves the ant makes.

**Reading w.** On steps where the ant follows smell (no target in view, one strongest smell, and the strongest smell is *not* its previous direction), the true chance of the smell move is (1 - 0.1) w + 0.1/4. So the w a model believes is (p_smell - 0.025) / 0.9.

In [ ]:
w_root = make_dataset("w")
wmix = {split: load_trait_split(w_root, split) for split in ["train", "val", "test"]}


def w_steps(states, prev):
    """Mask of 'trail decision' steps and the smell direction there. states (..., 19), prev (..., 4)."""
    shape = states.shape[:-1]
    s, p = states.reshape(-1, 19), prev.reshape(-1, 4)
    best = best_dir(s)
    m = (~sees_target(s)) & (n_ties(s) == 1) & (p.sum(1) > 0) & (p.argmax(1) != best)
    return m.reshape(shape), best.reshape(shape)


# Check the data: on trail-decision steps, follow rate = 0.9 w + 0.025
rows = []
for r in wmix["train"]:
    m, best = w_steps(r["state"], r["prev_action"])                 # (T, N)
    follow = r["action"] == best
    for i, w in enumerate(r["ant_trait"]):
        if m[:, i].any():
            rows.append(dict(w=round(float(w), 1), follows=follow[:, i][m[:, i]].mean(), steps=m[:, i].sum()))
wcheck = pd.DataFrame(rows).groupby("w").agg(follows=("follows", "mean"), steps=("steps", "mean"))
wcheck["expected"] = (1 - EPS_W) * wcheck.index + EPS_W / 4
print(wcheck.round(3))

fig, ax = plt.subplots(figsize=(6.5, 4.2))
ax.plot(wcheck.index, wcheck.expected, ls="--", color=C["gray"], lw=1.2, label="expected 0.9 w + 0.025")
ax.plot(wcheck.index, wcheck.follows, "o", color=C["blue"], label="measured, per ant")
ax.set(xlabel="the ant's own w", ylabel="follows the trail (decision steps)", title="Ants trust trails at their own w")
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
@torch.no_grad()
def collect_full(model, data, n_windows=N_VAL_WINDOWS, seed=0, batch=256):
    """Like collect_seq, but also returns the full input (we need the previous move)."""
    X, Y, EPS = data
    ants, t0 = all_windows(len(X), X.shape[1])
    keep = np.random.default_rng(seed).choice(len(ants), size=min(n_windows, len(ants)), replace=False)
    probs, xs, trait = [], [], []
    for start in range(0, len(keep), batch):
        x, _, e = get_batch(X, Y, EPS, ants[keep[start:start + batch]], t0[keep[start:start + batch]])
        probs.append(model(x.to(DEVICE)).softmax(-1).cpu().numpy())
        xs.append(x.numpy())
        trait.append(e)
    return np.concatenate(probs), np.concatenate(xs), np.concatenate(trait)


def implied_w(probs, x):
    """Per step: the w a model believes (W, L) and the decision-step mask (W, L)."""
    m, best = w_steps(x[..., :19], x[..., 19:23])
    p = np.take_along_axis(probs, best[..., None], axis=-1)[..., 0]
    return np.clip((p - EPS_W / 4) / (1 - EPS_W), 0, 1), m


def implied_w_windows(probs, x, sl=LATE):
    e, m = implied_w(probs, x)
    e, m = e[:, sl], m[:, sl]
    n = m.sum(1)
    return np.where(n > 0, (e * m).sum(1) / np.maximum(n, 1), np.nan)


seq_w = {split: trait_sequences(wmix[split], with_actions=True) for split in ["val", "test"]}
path = OUT_DIR / "seq_A_w.pt"
if path.exists():
    model_w = load_seq_model("A_w")
else:
    seed_everything(0)
    train = trait_sequences(wmix["train"], with_actions=True)
    model_w = AntGPT(d_in=23).to(DEVICE)
    train_seq_model(model_w, train)
    save_seq_model(model_w, "A_w", d_in=23)
    del train
wmix.pop("train")                                               # free memory

probs_w, x_w, w_true = collect_full(model_w, seq_w["val"])
e_w, m_w = implied_w(probs_w, x_w)
rows = []
for v in EPS_GRID:
    sel = np.isclose(w_true, v)
    row = {"true_w": v}
    for name, sl in BUCKETS.items():
        vals = e_w[sel][:, sl][m_w[sel][:, sl]]
        row[name] = vals.mean() if vals.size else np.nan
    rows.append(row)
w_by = pd.DataFrame(rows).set_index("true_w")
print("implied w by true w, per history range:")
print(w_by.round(3))
slope_w, icpt_w = fit_line(w_by.index, w_by["64-127"])
slope_w0, _ = fit_line(w_by.index, w_by["0"])
err_w = np.nanmean(np.abs(w_by["64-127"] - w_by.index))
print(f"\nafter 64-127 steps: slope {slope_w:.3f}, intercept {icpt_w:.3f}, mean error {err_w:.3f} | history-0 slope {slope_w0:.3f}")
RESULTS["T6 w reading: error / slope late / slope history 0"] = (err_w, slope_w, slope_w0)

In [ ]:
# Probes and the injection knob for w (same recipe as test 2, with the w readout)
val_hw = collect_hidden(model_w, seq_w["val"], seed=0)
test_hw = collect_hidden(model_w, seq_w["test"], seed=1)
probe_w = pd.DataFrame({stage: [r2(test_hw["eps"], ridge_fit(val_hw["hidden"][k][:, t].astype(np.float64),
                                                              val_hw["eps"].astype(np.float64))(
                                         test_hw["hidden"][k][:, t].astype(np.float64)))
                                 for t in PROBE_POSITIONS]
                        for k, stage in enumerate(STAGES)}, index=PROBE_POSITIONS).T
print("probe R² for w (test):")
print(probe_w.round(3))
means_w = {k: eps_means(val_hw["hidden"][k], val_hw["eps"]) for k in range(len(STAGES))}


def reading_w(model, data, stage=None, vec=None, value=0.5, seed=0):
    x, _ = windows_for_eps(data, value, seed=seed)
    p = run_with_injection(model, x, stage, vec)
    return safe_nanmean(implied_w_windows(p, x.cpu().numpy())), p, x


goals_w_val = {v: reading_w(model_w, seq_w["val"], value=v, seed=10)[0] for v in EPS_GRID}
goals_w = {v: reading_w(model_w, seq_w["test"], value=v, seed=0)[0] for v in EPS_GRID}


def knob_err_w(stage, alpha, data, goals, seed):
    return np.nanmean([abs(reading_w(model_w, data, stage, alpha * (means_w[stage][t] - means_w[stage][0.5]), 0.5, seed)[0] - goals[t])
                       for t in TARGETS])


stage_err_w = {k: knob_err_w(k, 2.0, seq_w["val"], goals_w_val, 10) for k in (1, 2, 3)}
stage_w = min(stage_err_w, key=stage_err_w.get)
alpha_err_w = {a: knob_err_w(stage_w, a, seq_w["val"], goals_w_val, 10) for a in ALPHA_GRID}
alpha_w = min(alpha_err_w, key=alpha_err_w.get)

rng = np.random.default_rng(0)
rows = []
for t in EPS_GRID:
    v = alpha_w * (means_w[stage_w][t] - means_w[stage_w][0.5])
    rows.append(dict(target=t, goal=goals_w[t], injected=reading_w(model_w, seq_w["test"], stage_w, v)[0],
                     random=reading_w(model_w, seq_w["test"], stage_w, random_like(v, rng))[0]))
knob_w = pd.DataFrame(rows).set_index("target")
on = knob_w.loc[TARGETS]
err_knob_w, err_rand_w = np.abs(on.injected - on.goal).mean(), np.abs(on.random - on.goal).mean()
print(f"\nw knob: stage {stage_w}, strength {alpha_w} (both chosen on val) | test error {err_knob_w:.3f}, random {err_rand_w:.3f}")
print(knob_w.round(3))
RESULTS[f"T6 w probe R2 layer3 t127"] = probe_w.iloc[-1, -1]
RESULTS[f"T6 w knob error / random (stage {stage_w}, strength {alpha_w})"] = (err_knob_w, err_rand_w)

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.6))
ax = axes[0]
diagonal(ax)
for name, color in zip(BUCKETS, RAMP):
    ax.plot(w_by.index, w_by[name], "-o", color=color, label=f"history {name}")
ax.set(xlabel="true w (the ant's own)", ylabel="implied w", title="Reading trail trust from history")
ax.legend(fontsize=8)
ax = axes[1]
for stage, color in zip(STAGES, RAMP[1:]):
    ax.plot(PROBE_POSITIONS, probe_w.loc[stage].values, "-o", color=color, label=stage)
ax.set(xlabel="steps of history seen", ylabel="probe R² for w (test)", ylim=(-0.1, 1), title="Where w is readable")
ax.legend(fontsize=8)
ax = axes[2]
ax.plot(knob_w.index, knob_w.goal, "-o", color=C["aqua"], label="goal (natural reading)")
ax.plot(knob_w.index, knob_w.injected, "-o", color=C["blue"], label=f"w 0.5 + w vector (stage {stage_w}, x{alpha_w})")
ax.plot(knob_w.index, knob_w.random, "--s", color=C["orange"], label="w 0.5 + random vector")
ax.set(xlabel="target w", ylabel="implied w (test)", ylim=(0, 1.05), title=f"Setting w: error {err_knob_w:.3f} (random {err_rand_w:.3f})")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

**How to read this.** A pass needs three things together: a history-0 line that is flat (no leak, since colonies are mixed), a late line that climbs toward the diagonal (w read from behavior), and a knob that tracks the goal clearly better than the random vector. Because w changes *which* move is made, a pass here cannot be explained by counting rule breaks, which answers the main objection to model B.

*Analysis: fill in after the run.*

## 7. Summary of all tests (copy into the first cell)

In [ ]:
def fmt(v):
    if isinstance(v, tuple):
        return " / ".join(f"{x:.3f}" if isinstance(x, (float, np.floating)) and np.isfinite(x) else str(x) for x in v)
    return f"{v:.3f}" if isinstance(v, (float, np.floating)) else str(v)


for k, v in RESULTS.items():
    print(f"{k:72s} {fmt(v)}")

print("\n--- pass / fail ---")
t1 = results1
leak_gone = abs(fit_line(t1["A_mix"][1].index, t1["A_mix"][1]["0"])[0]) < 0.1
behavior = t1["A_mix"][2]["eps error, eps<=0.7"] < 0.8 * t1["D_mix"][2]["eps error, eps<=0.7"]
print(f"T1 leak removed (A_mix history-0 slope < 0.1):            {leak_gone}")
print(f"T1 behavior matters (A_mix error < 0.8 x D_mix error):       {behavior}")
a = analyses["A_mix"]
print(f"T2 A_mix knob beats random by 2x:                            {a['test_err'] < 0.5 * a['random_err']}")
print(f"T3 bigger/two-step A beats A_mix on eps error:               "
      f"{min(results3['A_big'][2]['eps error, eps<=0.7'], results3['A_2step'][2]['eps error, eps<=0.7']) < results3['A_mix'][2]['eps error, eps<=0.7']}")
best_b = readouts.loc["B"].drop("ceiling (model output)").max()
print(f"T4 a readout reaches 90% of B's ceiling:                     {best_b >= 0.9 * readouts.loc['B', 'ceiling (model output)']}")
reach = all(abs(opt_table.loc[(t, 'optimized'), 'implied'] - goals_test_B[t]) < 0.05 for t in [0.1, 1.0])
print(f"T5 optimized vector reaches both extremes (within 0.05):     {reach}")
print(f"T6 w read (slope > 0.5, history-0 slope < 0.1) and set (2x): "
      f"{slope_w > 0.5 and abs(slope_w0) < 0.1 and err_knob_w < 0.5 * err_rand_w}")

**Overall analysis.** *Fill in after the run, using the printed summary.* A useful structure:
1. Did removing the colony leak (test 1) keep the eps reading alive? This decides whether RQ1 holds.
2. Do probes and the knob work on models that learned without the rule (tests 2 and 6)? This decides whether RQ3 and RQ4 hold beyond counting.
3. What limits the reading: evidence (test 3, ideal observer), readout (test 4), or what the layer can express (test 5)?